In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Callable, Optional
from dataclasses import dataclass

In [2]:
# =========================
# 4-easy-step-reuse
# =========================
files = {
"4-easy-step-reuse_0_stress_run1": "../data/stressed_1000_baremetal_2026_02_08_23_39_58_73e3f926/data",
"4-easy-step-reuse_0_stress_run2": "../data/stressed_1000_baremetal_2026_02_08_23_42_56_f736da0f/data",
"4-easy-step-reuse_0_stress_run3": "../data/stressed_1000_baremetal_2026_02_08_23_45_51_9d659069/data",

"4-easy-step-reuse_5_stress_run1": "../data/stressed_1000_baremetal_2026_02_08_23_48_49_761a0e42/data",
"4-easy-step-reuse_5_stress_run2": "../data/stressed_1000_baremetal_2026_02_08_23_51_43_efaebfb1/data",
"4-easy-step-reuse_5_stress_run3": "../data/stressed_1000_baremetal_2026_02_08_23_54_37_33c88927/data",

"4-easy-step-reuse_50_stress_run1": "../data/stressed_1000_baremetal_2026_02_08_23_57_32_1fc54db8/data",
"4-easy-step-reuse_50_stress_run2": "../data/stressed_1000_baremetal_2026_02_09_00_00_24_b967563e/data",
"4-easy-step-reuse_50_stress_run3": "../data/stressed_1000_baremetal_2026_02_09_00_03_21_c26374c7/data",

# =========================
# 4-step-reuse
# =========================
"4-step-reuse_0_stress_run1": "../data/stressed_1000_baremetal_2026_02_09_00_06_11_b75eebc0/data",
"4-step-reuse_0_stress_run2": "../data/stressed_1000_baremetal_2026_02_09_00_07_52_f6fdcb6f/data",
"4-step-reuse_0_stress_run3": "../data/stressed_1000_baremetal_2026_02_09_00_09_32_07d2a771/data",

"4-step-reuse_5_stress_run1": "../data/stressed_1000_baremetal_2026_02_09_00_11_23_e99b3121/data",
"4-step-reuse_5_stress_run2": "../data/stressed_1000_baremetal_2026_02_09_00_13_04_bb64c444/data",
"4-step-reuse_5_stress_run3": "../data/stressed_1000_baremetal_2026_02_09_00_14_44_20ed032d/data",

"4-step-reuse_50_stress_run1": "../data/stressed_1000_baremetal_2026_02_09_00_16_25_b896f2bc/data",
"4-step-reuse_50_stress_run2": "../data/stressed_1000_baremetal_2026_02_09_00_18_05_46dc0205/data",
"4-step-reuse_50_stress_run3": "../data/stressed_1000_baremetal_2026_02_09_00_19_46_a21425af/data",

# =========================
# baseline
# =========================
"baseline_0_stress_run1": "../data/stressed_1000_baremetal_2026_02_09_00_21_27_395ca0d8/data",
"baseline_0_stress_run2": "../data/stressed_1000_baremetal_2026_02_09_00_24_22_9b3ef12d/data",
"baseline_0_stress_run3": "../data/stressed_1000_baremetal_2026_02_09_00_27_16_637dd1e6/data",

"baseline_5_stress_run1": "../data/stressed_1000_baremetal_2026_02_09_00_30_11_1fba75df/data",
"baseline_5_stress_run2": "../data/stressed_1000_baremetal_2026_02_09_00_33_06_df00cb12/data",
"baseline_5_stress_run3": "../data/stressed_1000_baremetal_2026_02_09_00_35_56_a5d1650f/data",

"baseline_50_stress_run1": "../data/stressed_1000_baremetal_2026_02_09_00_38_50_e1938452/data",
"baseline_50_stress_run2": "../data/stressed_1000_baremetal_2026_02_09_00_41_40_cb353878/data",
"baseline_50_stress_run3": "../data/stressed_1000_baremetal_2026_02_09_00_44_33_7ba8f906/data",

# =========================
# full-reuse
# =========================
"full-reuse_0_stress_run1": "../data/stressed_1000_baremetal_2026_02_09_00_47_27_e72e3623/data",
"full-reuse_0_stress_run2": "../data/stressed_1000_baremetal_2026_02_09_00_49_08_4cdf1ac6/data",
"full-reuse_0_stress_run3": "../data/stressed_1000_baremetal_2026_02_09_00_50_49_cea031f3/data",

"full-reuse_5_stress_run1": "../data/stressed_1000_baremetal_2026_02_09_00_52_29_4f806d3d/data",
"full-reuse_5_stress_run2": "../data/stressed_1000_baremetal_2026_02_09_00_54_10_b17f7312/data",
"full-reuse_5_stress_run3": "../data/stressed_1000_baremetal_2026_02_09_00_55_50_a2ccbb52/data",

"full-reuse_50_stress_run1": "../data/stressed_1000_baremetal_2026_02_09_00_57_31_9b60001f/data",
"full-reuse_50_stress_run2": "../data/stressed_1000_baremetal_2026_02_09_00_59_12_64fb4538/data",
"full-reuse_50_stress_run3": "../data/stressed_1000_baremetal_2026_02_09_01_00_52_2e7588d8/data",
}

In [4]:
import os
import pandas as pd

# =========================
# Flatten helper function
# =========================
def flatten_dict(d, parent_key='', sep='_'):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        elif isinstance(v, list):
            continue
        else:
            items.append((new_key, v))
    return dict(items)

run_data_list = []

# Will hold metric name -> combined DataFrame
experiment_data = {}

# =========================
# Main loop: process folders
# =========================
for key, folder_path in files.items():
    experiment, stress, _, run = key.split("_")
    stress_level = int(stress)
    run_number = int(run[-1])
    
    # 1️⃣ Collect all CSV files in this folder (each CSV = a metric)
    csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]
    
    # 2️⃣ Create run_data metadata
    run_data = flatten_dict({
        "unique_id": key,
        "experiment": experiment,
        "stress": stress_level,
        "run": run_number
    })
    run_data_list.append(run_data)
    
    # 3️⃣ Process each CSV (metric)
    for csv_file in csv_files:
        metric_name = os.path.splitext(csv_file)[0]  # filename without .csv
        df = pd.read_csv(os.path.join(folder_path, csv_file))
        
        # Add run_data metadata to this metric
        for run_key, run_value in run_data.items():
            df[run_key] = run_value
        
        # Concatenate to experiment_data
        if metric_name not in experiment_data:
            experiment_data[metric_name] = df
        else:
            experiment_data[metric_name] = pd.concat([experiment_data[metric_name], df], ignore_index=True)

# 4️⃣ Convert run_data_list to DataFrame
run_data = pd.DataFrame(run_data_list)

In [23]:


@dataclass
class ExperimentAggregator:
    kind: str
    experiment_grouping: list[str]
    per_run_grouping: list[str]
    filter: Optional[Callable[[pd.DataFrame], pd.Series]] = None
    per_run_total: Optional[Callable[[pd.Series], float]] = None
    per_run_name : Optional[Callable[[pd.Series], pd.Series]] = None

#TODO: check with SM if correct    
experiment_nodes = ["gisele", "sm-gpu"]
experiment_namespaces = ["pipeline"]

def filter_experiment_nodes(df: pd.DataFrame) -> pd.Series:
    return df[df["node"].isin(experiment_nodes)]

def filter_experiment_namespaces(df: pd.DataFrame) -> pd.Series:
    return df[df["namespace"].isin(experiment_namespaces)]

def delta_per_run(series: pd.Series) -> float:
    return series.max() - series.min()

def time_series_integral(series: pd.Series, step_size: int = 10) -> float:
    return np.trapz(series.values, dx=step_size)
# parallel-full-data-pipelinewxhfd-call-generate-name-service-1957051688

import re
pipeline = re.compile(r"parallel-full-data-pipeline[a-z0-9]+-([\w-]+)-parallel-full-data-pipeline-[0-9]+")
pipeline_service = re.compile(r"parallel-full-data-pipeline[a-z0-9]+-([\w-]+)-service-[0-9]+")
knative = re.compile(r"([\w-]+)-knative-([0-9]+).*", re.IGNORECASE | re.DOTALL)
generic = re.compile(r"([\w-]+)-[a-z0-9]+-[a-z0-9]+")

def pod_name_extractor( key: str) -> Callable[[pd.Series],pd.Series]:
    def name_extractor(series: pd.Series) -> pd.Series:
        def extract_name(pod_name: str) -> str:
            if pod_name.startswith("parallel-full-data-pipeline"):
                match = pipeline.match(pod_name)
                if match:
                    return match.group(1)
                else:
                    match = pipeline_service.match(pod_name)
                    if match:
                        return match.group(1)+"-argo"
                    
            elif "knative" in pod_name:
                match = knative.match(pod_name)
                if match:
                    return match.group(1)+"-knative"
                    
            else:
                match = generic.match(pod_name)
                if match:
                    return match.group(1)
            raise ValueError(f"No match found for pod name: {pod_name}")
        
        return series[key].apply(extract_name)

    return name_extractor


experiment_data_aggregators = {
    "kepler_node_cpu_power_raw":
        ExperimentAggregator(kind="counter", experiment_grouping=["experiment", "stress","runHash"], per_run_grouping=["node"], filter = filter_experiment_nodes, per_run_total=delta_per_run),
    "kepler_pod_cpu_power_joules_raw":ExperimentAggregator(kind="counter", experiment_grouping=["experiment", "stress","runHash"], per_run_grouping=["pod_id"], per_run_total=delta_per_run, per_run_name=pod_name_extractor(key="pod_name")),
    "pod_network_receive_bytes_raw":ExperimentAggregator(kind="counter", experiment_grouping=["experiment", "stress","runHash"], per_run_grouping=["pod"], per_run_total=delta_per_run, per_run_name=pod_name_extractor(key="pod")),
    "pod_network_transmit_bytes_raw": ExperimentAggregator(kind="counter", experiment_grouping=["experiment", "stress","runHash"], per_run_grouping=["pod"], per_run_total=delta_per_run, per_run_name=pod_name_extractor(key="pod")),
    "system_cpu_usage_raw": ExperimentAggregator(kind="counter", experiment_grouping=["experiment", "stress","runHash"], per_run_grouping=["pod","name"], filter=filter_experiment_namespaces, per_run_total=delta_per_run, per_run_name=pod_name_extractor(key="pod")),
    "wall_power":ExperimentAggregator(kind="gauge", experiment_grouping=["experiment", "stress","runHash"], per_run_grouping=["node"], filter = filter_experiment_nodes, per_run_total=time_series_integral ),
    "pod_memory_usage":ExperimentAggregator(kind="gauge", experiment_grouping=["experiment", "stress","runHash"], per_run_grouping=["pod"], per_run_total=time_series_integral, per_run_name=pod_name_extractor(key="pod")),
}

for metric, aggregator in experiment_data_aggregators.items():
    data = experiment_data[metric]
    if aggregator.per_run_name:
        data["worker_name"] = aggregator.per_run_name(data)

In [24]:
def get_query(cnf: Config, metric: str) -> str:
    if metric not in cnf.measurements:
        raise ValueError(f"Metric {metric} not found in configuration.")
    return cnf.measurements[metric].query

def print_queries(cnf: Config):
    for metric in cnf.measurements.keys():
        query = get_query(cnf, metric)
        print(f"{metric}: {query}")

for key in experiment_data_aggregators.keys():
    print(f"{key}: {get_query(cnf, key)}")

NameError: name 'Config' is not defined

Metrics we want to look at:
 - argo_workflows_completed_total_raw
 - argo_workflows_k8s_requests_raw
 - argo_workflows_operation_duration_seconds_raw
 - argo_workflows_pods_total_raw
 - argo_workflows_queue_adds_raw
 - argo_workflows_total_count_raw
 - argo_workflows_workers_busy_raw
 - kepler_node_cpu_power_raw
 - kepler_pod_cpu_power_joules_raw
 - pod_network_receive_bytes_raw
 - pod_network_transmit_bytes_raw
 - system_cpu_usage_raw

* We need to validate some base assumptions:
 - That the aggregations we are using are correct (e.g. sum vs avg)
 - That the grouping we are using is correct (e.g. by pod, by node, by workflow)
 - That experimentes are executed as expected, e.g., all pods are on the namespaces we consider for the experiments...

In [6]:
def run_aggregate(key):
    if key not in experiment_data_aggregators:
        print(f"Metric {key} does not have an aggregator defined, skipping.")
        return None

    grouping = experiment_data_aggregators[key].experiment_grouping + (experiment_data_aggregators[key].per_run_grouping if experiment_data_aggregators[key].per_run_grouping is not None else [])
    
    agg = experiment_data_aggregators[key].per_run_total
    if agg is None:
        print(f"Metric {key} does not have a per run total function defined, using sum as default.")
        return None
    # per run we first aggregate each component (e.g. node, pod) and then we sum them up to get the total per run, then we can aggregate by experiment and stress level to get the mean and stddev across runs
    if experiment_data_aggregators[key].filter is not None:
        data = experiment_data_aggregators[key].filter(experiment_data[key])
    else:
        data = experiment_data[key]
    
    data = data.groupby(grouping)[key].agg(agg).groupby(experiment_data_aggregators[key].experiment_grouping).sum().reset_index(name="total")
    return data
    

In [ ]:
intersting_metrics = ["kepler_node_cpu_power_raw", "kepler_pod_cpu_power_joules_raw","pod_network_receive_bytes_raw","pod_network_transmit_bytes_raw","system_cpu_usage_raw", "pod_memory_usage", "wall_power"]


log_scale = False
for key in intersting_metrics:
    data = run_aggregate(key)
    min = data["total"].min()
    max = data["total"].max()
    mean = data["total"].mean()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(data=data, x="experiment", y="total",ax=ax, order=view_order)
    ax.set_title(f"Average {key} by Experiment")
    ax.set_xlabel("Experiment")
    ax.set_ylabel(f"Average Total {key}")
    if log_scale:
        ax.set_yscale("log") 
    # else:
        # ax.set_ylim(min*0.95, max*1.05)

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(data=data, x="experiment", y="total", hue="stress", hue_order=[0,5,50], order=view_order, ax=ax)
    ax.set_title(f"Average {key} by Experiment")
    ax.set_xlabel("Experiment")
    ax.set_ylabel(f"Average Total {key}")
    # ax.set_yscale("log") 
    # ax.set_ylim(min*0.95, max*1.05)

In [ ]:
wall_data = run_aggregate("wall_power")
node_data = run_aggregate("kepler_node_cpu_power_raw")
pod_data = run_aggregate("kepler_pod_cpu_power_joules_raw")

# Wall Power >> Node Power >> Pod Power (staked bar chart with error bars for each experiment and stress level)
merged_data = wall_data.merge(node_data, on=["experiment", "stress", "runHash"], suffixes=("_wall", "_node"))
merged_data = merged_data.merge(pod_data, on=["experiment", "stress", "runHash"], suffixes=("", "_pod"))

fig, ax = plt.subplots(figsize=(10, 6))



sns.barplot(data=merged_data, x="experiment", y="total_wall", order=view_order, ax=ax, color="red", alpha=1, label="Wall Power")
sns.barplot(data=merged_data, x="experiment", y="total_node", order=view_order, ax=ax, color="blue", alpha=0.7, label="Node Power")
sns.barplot(data=merged_data, x="experiment", y="total", order=view_order, ax=ax, alpha=1, label="Pod Power")

merged_data


In [ ]:
run_data[["experiment", "stress","runHash", "duration"]].groupby(["experiment", "stress"])["duration"].agg(["mean", "std"]).reset_index()

In [ ]:
intersting_metrics = ["kepler_node_cpu_power_raw", "kepler_pod_cpu_power_joules_raw","pod_network_receive_bytes_raw","pod_network_transmit_bytes_raw","system_cpu_usage_raw"]

for key in intersting_metrics:
    
    data = run_aggregate(key)
    data = pd.merge(data.reset_index(), run_data[["experiment", "stress","runHash", "duration"]], on=["experiment", "stress","runHash"])
    data[f"{key}_per_second"] = data["total"] / data["duration"]

    data = data.groupby(["experiment", "stress"])[f"{key}_per_second"].agg(["mean", "std"]).reset_index()
    # data = data[data["stress"]==0]
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.barplot(data=data, x="experiment", y="mean", hue="stress", hue_order=[0,5,50], order=["baseline", "4-easy-step-reuse", "4-step-reuse", "full-reuse"],ax=ax)
    ax.set_title(f"Average {key} per Second by Experiment")
    ax.set_xlabel("Experiment")
    ax.set_ylabel(f"Average {key} per Second")


In [ ]:
data = experiment_data["kepler_pod_cpu_power_joules_raw"].groupby(["experiment", "stress", "runHash", "worker_name", "pod_id"])[["kepler_pod_cpu_power_joules_raw"]].agg(delta_per_run).groupby(["experiment", "stress", "worker_name"]).mean().reset_index()

# turn experiment_stress into a single column with experiment and stress level separated by an underscore
data["experiment_stress"] = data["experiment"] + "_" + data["stress"].astype(str)

# stack worker_name values instead of using hues
stacked = data.pivot(index="experiment_stress", columns="worker_name", values="kepler_pod_cpu_power_joules_raw").fillna(0)

# build a large, distinguishable palette with lightness variation
import builtins
import colorsys
import matplotlib.colors as mcolors

def adjust_lightness(color, factor):
    r, g, b = mcolors.to_rgb(color)
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    l = builtins.max(0.15, builtins.min(0.85, l * factor))
    r2, g2, b2 = colorsys.hls_to_rgb(h, l, s)
    return (r2, g2, b2)

n_colors = stacked.shape[1]
base = sns.color_palette("Pastel2", 27)

palette = []
for i in range(n_colors):
    color = base[i % len(base)]
    # alternate light/dark for extra separation
    factor = 0.85 if (i % 2 == 0) else 1.15
    palette.append(adjust_lightness(color, factor))

fig, ax = plt.subplots(figsize=(12, 10))

stacked.plot(kind="bar", stacked=True, ax=ax, color=palette)

# apply hatch patterns per worker (per hue) consistently across all stacks
hatch_patterns = ['', '//', '\\\\', '||', '--', '++', 'xx', '**', 'OO', 'o', '...']
for container_idx, container in enumerate(ax.containers):
    hatch = hatch_patterns[container_idx % len(hatch_patterns)]
    for patch in container:
        patch.set_hatch(hatch)
        patch.set_edgecolor('black')
        patch.set_linewidth(0.5)

ax.set_xlabel("Experiment_Stress")
ax.set_ylabel("kepler_pod_cpu_power_joules_raw")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment="right")
ax.legend(title="Worker Name", bbox_to_anchor=(1.05, 1), loc="upper left")